In [ ]:
import pandas as pd
dat = pd.read_csv('human_sentences.csv')

In [2]:
from datasets import load_dataset

ds = load_dataset("sedthh/gutenberg_english")

/data/sbhate/envs/spatial_tx/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
Generating train split: 100%|██████████| 48284/48284 [00:43<00:00, 1098.37 examples/s]


In [71]:
def classify_book(subjects):
    """
    Classify book into science/philosophy or novels
    Returns: 'science_philosophy', 'novel', or None (ignore)
    """
    if not subjects:
        return None
    
    subjects_lower = subjects.lower()
    
    # Science/Philosophy keywords
    sci_phil_keywords = [
        'science', 'philosophy', 'physics', 'chemistry', 'biology',
        'mathematics', 'astronomy', 'psychology', 'logic', 'ethics',
        'metaphysics', 'epistemology', 'natural history', 'geology',
        'evolution', 'scientific', 'mathematics', 'algebra', 'geometry',
        'medicine', 'anatomy', 'botany', 'zoology', 'technology',
        'engineering', 'political science', 'economics', 'sociology'
    ]
    
    # Novel keywords
    novel_keywords = [
        'fiction', 'novel', 'romance', 'adventure', 'mystery',
        'detective', 'gothic', 'love stories', 'short stories',
        'fantasy', 'science fiction', 'historical fiction', 'thriller',
        'western', 'sea stories', 'war stories', 'humorous stories'
    ]
    
    # Check for matches
    has_sci_phil = any(kw in subjects_lower for kw in sci_phil_keywords)
    has_novel = any(kw in subjects_lower for kw in novel_keywords)
    
    # Exclude if both or neither
    if has_sci_phil and not has_novel:
        return 'science_philosophy'
    elif has_novel and not has_sci_phil:
        return 'novel'
    elif has_novel:  # If both, prefer novel (more common)
        return 'novel'
    
    return None

In [ ]:
from tqdm import tqdm
import ast

In [64]:
author_dates = {}
for i in tqdm(range(len(ds['train']))):
    year = ast.literal_eval(ds['train'][i]['METADATA'])['authors'].split(';')[0][-4:]
    author_dates[i]  = year

100%|██████████| 48284/48284 [00:13<00:00, 3629.31it/s]


In [74]:
subjects = {}
for i in tqdm(range(len(ds['train']))):
    subjects[i] = classify_book(ast.literal_eval(ds['train'][i]['METADATA'])['subjects'])

100%|██████████| 48284/48284 [00:13<00:00, 3598.32it/s]


In [84]:
text_ids = {}
for i in tqdm(range(len(ds['train']))):
    text_ids[i] = ast.literal_eval(ds['train'][i]['METADATA'])['text_id']

100%|██████████| 48284/48284 [00:13<00:00, 3631.98it/s]


In [97]:
deepmind_ds = pd.read_csv('metadata.csv',header = None,index_col = 0)
comb = pd.concat([pd.Series(text_ids).reset_index().set_index(0),deepmind_ds,],axis=1).dropna()



In [107]:
pd.DataFrame({'class':subjects}).dropna()['class'].value_counts()

class
novel                 19015
science_philosophy     2449
Name: count, dtype: int64

In [113]:
pd.merge(comb, pd.DataFrame({'class':subjects}).dropna(), left_on ='index', right_index = True).rename({'index':'ds_index'},axis=1).to_pickle('ds_metadata.pkl')

In [167]:
df = pd.read_csv('human_sentences.csv')
ds_metadata = pd.read_pickle('ds_metadata.pkl')
df = df[(df['dataset_index'].isin(ds_metadata['ds_index'])) & (df['sentence'].str.len() < 600)]